In [1]:
"""VSS setup — shared with `dhl_dashboard/vss_client.py` (token file, 10082 backoff)."""
from __future__ import annotations

import importlib
import json
import os
import sys
from pathlib import Path

# --- Edit if your server/account differs ---
BASE_URL = "http://40.76.130.233:9966"
USERNAME = "mawa@controltech-ea.com"
PASSWORD_PLAINTEXT = "Kenya+123"

# Set True, re-run this cell → fresh apiLogin (skips .vss_token.txt + kernel cache).
# Saves the new token to .vss_token.txt. If VSS returns 10082, wait ~10 min and retry.
FORCE_FRESH_LOGIN = False


def _resolve_dashboard_dir() -> str:
    """This notebook lives in dhl_dashboard next to vss_client.py."""
    candidates: list[Path] = []
    env = os.environ.get("DHL_DASHBOARD_DIR", "").strip()
    if env:
        candidates.append(Path(env).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    if cwd.name != "dhl_dashboard":
        candidates.append(cwd / "dhl_dashboard")
    seen: set[Path] = set()
    for p in candidates:
        if p in seen or not p.is_dir():
            continue
        seen.add(p)
        if (p / "vss_client.py").is_file():
            return str(p)
    raise RuntimeError(
        "Cannot find vss_client.py. Set env DHL_DASHBOARD_DIR to this folder, "
        "or start Jupyter with working directory = dhl_dashboard."
    )


DASHBOARD_DIR = _resolve_dashboard_dir()
if DASHBOARD_DIR not in sys.path:
    sys.path.insert(0, DASHBOARD_DIR)

os.environ["VSS_BASE_URL"] = BASE_URL
os.environ["VSS_USERNAME"] = USERNAME
os.environ["VSS_PASSWORD"] = PASSWORD_PLAINTEXT

import vss_client

importlib.reload(vss_client)

from vss_client import (
    BASE_URL,
    USERNAME,
    current_gps_and_status,
    ensure_token,
    last_vss_token_source,
    md5_hex,
    vss_post,
    vss_post_raw,
)

# Fleet/form fallback in later cells uses this Requests session (same as dashboard).
session = vss_client._session

# Demo IDs (optional)
DEVICE_ONE = "99990001"
DEVICES_MANY = ["99990001", "99990002"]

if "<SERVER_IP>" in BASE_URL:
    raise ValueError('Set BASE_URL, e.g. BASE_URL="http://192.168.1.2:9966"')


def online_status_single(token: str, device_id: str):
    j = vss_post(
        "/vss/onoffline/queryDeviceStatus.action",
        {"token": token, "deviceID": device_id},
    )
    return j["data"]


def online_status_multiple(token: str, device_ids: list[str]):
    j = vss_post(
        "/vss/onoffline/queryMoreDeviceStatus.action",
        {"token": token, "deviceID": ",".join(device_ids)},
    )
    return j["data"]


if FORCE_FRESH_LOGIN:
    os.environ.setdefault("VSS_10082_RETRY_LOGIN", "1")
    token, pid = ensure_token(force=True, skip_file=True, login_max_wait_seconds=600)
else:
    token, pid = ensure_token()

print("token:", token)
print("pid:", pid)
print(
    "token source:",
    last_vss_token_source(),
    "(memory=reuse this kernel; env=VSS_TOKEN; file=.vss_token.txt; login=apiLogin)",
)
if FORCE_FRESH_LOGIN:
    print("fresh login requested — token written to .vss_token.txt")


token: 8bb5a8d8cc0a4fafac310f33673fddef
pid: 9b61da86425a3921c2df028db367b5eda6886b27c975b8d3b633186356aa
token source: file (memory=reuse this kernel; env=VSS_TOKEN; file=.vss_token.txt; login=apiLogin)


In [ ]:
%pip install gspread google-auth pandas requests --quiet

import requests
import pandas as pd
import gspread
from google.oauth2 import service_account
from datetime import datetime, timezone
import json
import os
from pyspark.sql import SparkSession
#from ZoneInfo import ZoneInfo

# ─────────────────────────────────────────────────────────────
# FILE PATHS
# ─────────────────────────────────────────────────────────────
ACCOUNTS_JSON_PATH          = "/lakehouse/default/Files/accounts.json"
GOOGLE_CREDENTIALS_PATH     = "/lakehouse/default/Files/credentials.json"

# ─────────────────────────────────────────────────────────────
# MiX SETTINGS
# ─────────────────────────────────────────────────────────────
SERVER_KEY               = "mix_uk"
GROUP_IDS                = [3522792081055458809]
QUANTITY                 = 1
CACHED_SINCE             = None
ENSURE_REVERSE_GEOCODED  = True

# ─────────────────────────────────────────────────────────────
# GOOGLE SHEETS SETTINGS
# ─────────────────────────────────────────────────────────────
GOOGLE_SHEET_NAME  = "AssetPositions"   # ← Change to your exact sheet name
WORKSHEET_TAB      = "Live Positions"
SERVICE_ACCOUNT_EMAIL = "fleetpositions@menengai-489712.iam.gserviceaccount.com"

print("✅ Configuration loaded")
print(f"   MiX Server      : {SERVER_KEY}")
print(f"   Group IDs       : {GROUP_IDS}")
print(f"   Google Sheet    : {GOOGLE_SHEET_NAME} / {WORKSHEET_TAB}")
print(f"   Service Account : {SERVICE_ACCOUNT_EMAIL}")

def get_creds(path: str, server: str) -> dict:
    with open(path, "r") as f:
        all_creds = json.load(f)
    if server not in all_creds:
        raise KeyError(f"Key '{server}' not found. Available: {list(all_creds.keys())}")
    return all_creds[server]


def get_bearer_token(creds: dict) -> str:
    token_url = f"{creds['IdentityUrl']}/core/connect/token"
    scope     = creds["IdentityScope"].replace("+", " ")

    payload = {
        "grant_type"   : "password",
        "client_id"    : creds["IdentityClientId"],
        "client_secret": creds["IdentityClientSecret"],
        "username"     : creds["IdentityUsername"],
        "password"     : creds["IdentityPassword"],
        "scope"        : scope
    }

    print(f"🔐 Requesting MiX token from: {token_url}")
    resp = requests.post(token_url, data=payload, timeout=30)

    if resp.status_code == 200:
        data       = resp.json()
        expires_in = int(data.get("expires_in", 0))
        print(f"✅ Token acquired — valid for {expires_in // 3600}h {(expires_in % 3600) // 60}m")
        return data["access_token"]
    else:
        print(f"❌ Token request failed ({resp.status_code}): {resp.text}")
        resp.raise_for_status()


# ── Load and authenticate ──
mix_creds    = get_creds(ACCOUNTS_JSON_PATH, SERVER_KEY)
API_URL      = mix_creds["ApiUrl"]
BEARER_TOKEN = get_bearer_token(mix_creds)

print(f"   API Base URL : {API_URL}")

def get_latest_positions(api_url, bearer_token, group_ids,
                         quantity=1, cached_since=None,
                         ensure_reverse_geocoded=True) -> list:

    # Correct endpoint path from MiX UK Swagger
    url = f"{api_url}/api/positions/groups/latest/{quantity}"

    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type" : "application/json",
        "Accept"       : "application/json"
    }

    params = {}
    if cached_since:
        params["cachedSince"] = cached_since
    if ensure_reverse_geocoded is not None:
        params["ensureReverseGeocoded"] = str(ensure_reverse_geocoded).lower()

    print(f"📡 Calling MiX API: {url}")
    print(f"   Group IDs : {group_ids}")

    resp = requests.post(url, headers=headers, params=params, json=group_ids, timeout=30)
    print(f"   Status    : {resp.status_code}")

    if resp.status_code == 200:
        data = resp.json()
        print(f"✅ {len(data)} position record(s) returned")
        return data
    else:
        print(f"❌ API Error {resp.status_code}: {resp.text[:300]}")
        resp.raise_for_status()


raw_positions = get_latest_positions(
    api_url                 = API_URL,
    bearer_token            = BEARER_TOKEN,
    group_ids               = GROUP_IDS,
    quantity                = QUANTITY,
    cached_since            = CACHED_SINCE,
    ensure_reverse_geocoded = ENSURE_REVERSE_GEOCODED
)

print("\n📋 Sample record (first asset):")
print(json.dumps(raw_positions[0] if raw_positions else {}, indent=2))

In [ ]:
def safe_get(record: dict, *keys):
    for k in keys:
        if k in record:
            return record[k]
    return ""


def parse_positions(raw_data: list) -> pd.DataFrame:
    run_ts  = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    records = []

    for item in raw_data:
        records.append({
            "Asset ID"          : safe_get(item, "AssetId"),
            "Driver ID"         : safe_get(item, "DriverId"),
            "Latitude"          : safe_get(item, "Latitude"),
            "Longitude"         : safe_get(item, "Longitude"),
            "Speed (km/h)"      : safe_get(item, "SpeedKilometresPerHour"),
            "Heading"           : safe_get(item, "Heading"),
            "Altitude (m)"      : safe_get(item, "AltitudeMetres"),
            "Address"           : safe_get(item, "FormattedAddress"),
            "GPS Source"        : safe_get(item, "Source"),
            "Satellites"        : safe_get(item, "NumberOfSatellites"),
            "Event Time (UTC)"  : safe_get(item, "Timestamp"),
            "Last Updated"      : run_ts
        })

    return pd.DataFrame(records)


df_positions = parse_positions(raw_positions)
print(f"✅ Parsed {len(df_positions)} asset records")
display(df_positions.head(10))

In [ ]:
def try_asset_endpoints(api_url, bearer_token, group_id) -> dict:
    
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type" : "application/json",
        "Accept"       : "application/json"
    }

    # Known MiX asset endpoint variations to try
    candidates = [
        ("GET",  f"{api_url}/api/assets/groups/{group_id}"),
        ("GET",  f"{api_url}/api/assets/group/{group_id}"),
        ("POST", f"{api_url}/api/assets/groups"),
        ("GET",  f"{api_url}/api/assets?groupId={group_id}"),
        ("GET",  f"{api_url}/api/v1/assets/groups/{group_id}"),
        ("GET",  f"{api_url}/api/groups/{group_id}/assets"),
    ]

    for method, url in candidates:
        print(f"📡 {method} {url}")
        if method == "GET":
            resp = requests.get(url, headers=headers, timeout=30)
        else:
            resp = requests.post(url, headers=headers, json=[group_id], timeout=30)
        print(f"   Status: {resp.status_code}")
        if resp.status_code == 200:
            assets = resp.json()
            print(f"✅ SUCCESS — {len(assets)} assets found!")
            print(f"   Working URL: {url}")
            print("\n📋 Sample asset record:")
            print(json.dumps(assets[0] if assets else {}, indent=2))
            return {a.get("AssetId", a.get("assetId")): a for a in assets}
        print()

    print("❌ All attempts failed — check Swagger for correct Assets endpoint")
    return {}


asset_lookup = try_asset_endpoints(API_URL, BEARER_TOKEN, GROUP_IDS[0])
print(f"\n✅ Asset lookup ready — {len(asset_lookup)} entries")

from datetime import datetime, timezone, timedelta

# Kenya timezone = UTC+3
KENYA_TZ = timezone(timedelta(hours=3))

def safe_get(record: dict, *keys):
    for k in keys:
        if k in record:
            return record[k]
    return ""


def parse_positions(raw_data: list, asset_lookup: dict) -> pd.DataFrame:
    run_ts  = datetime.now(KENYA_TZ).strftime("%Y-%m-%d %H:%M:%S EAT")
    records = []

    for item in raw_data:
        asset_id   = safe_get(item, "AssetId")
        asset_info = asset_lookup.get(asset_id, {})

        records.append({
            "Asset Name"        : safe_get(asset_info, "Description"),
            "Registration"      : safe_get(asset_info, "RegistrationNumber"),
            "Make"              : safe_get(asset_info, "Make"),
            "Asset ID"          : asset_id,
            "Driver ID"         : safe_get(item, "DriverId"),
            "Latitude"          : safe_get(item, "Latitude"),
            "Longitude"         : safe_get(item, "Longitude"),
            "Speed (km/h)"      : safe_get(item, "SpeedKilometresPerHour"),
            "Heading"           : safe_get(item, "Heading"),
            "Altitude (m)"      : safe_get(item, "AltitudeMetres"),
            "Address"           : safe_get(item, "FormattedAddress"),
            "GPS Source"        : safe_get(item, "Source"),
            "Event Time (EAT)"  : safe_get(item, "Timestamp"),
            "Last Updated (EAT)": run_ts
        })

    return pd.DataFrame(records)


df_positions = parse_positions(raw_positions, asset_lookup)
print(f"✅ Parsed {len(df_positions)} asset records")
print(f"   Last Updated: {datetime.now(KENYA_TZ).strftime('%Y-%m-%d %H:%M:%S EAT')}")
display(df_positions.head(10))



In [ ]:
# ── Install libraries if not present ─────────────────────────────
!pip install gspread google-auth


# ── Imports ─────────────────────────────────────────────────────
import gspread
from google.oauth2.service_account import Credentials
from datetime import datetime
from zoneinfo import ZoneInfo


# ── Google API Scopes ───────────────────────────────────────────
scope = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]


# ── Authenticate with Service Account ───────────────────────────
print("🔐 Authenticating with Google Sheets API...")

creds = Credentials.from_service_account_file(
    "/lakehouse/default/Files/credentials.json",
    scopes=scope
)

client = gspread.authorize(creds)


# ── Open Spreadsheet & Worksheet ─────────────────────────────────
print("📄 Opening Google Sheet...")

sheet = client.open(GOOGLE_SHEET_NAME)

try:
    worksheet = sheet.worksheet(WORKSHEET_TAB)
except gspread.exceptions.WorksheetNotFound:
    print("Worksheet not found — creating it...")
    worksheet = sheet.add_worksheet(
        title=WORKSHEET_TAB,
        rows=5000,
        cols=20
    )


# ── Prepare Data ─────────────────────────────────────────────────
worksheet.clear()

KENYA_TZ = ZoneInfo("Africa/Nairobi")
last_run_eat = datetime.now(KENYA_TZ).strftime("%Y-%m-%d %H:%M:%S EAT")

header = df_positions.columns.tolist()
rows   = df_positions.fillna("").astype(str).values.tolist()


# ── Row 1 : Refresh Timestamp ───────────────────────────────────
worksheet.update(
    "A1",
    [[f"Last Refreshed: {last_run_eat} | Total Assets: {len(rows)}"]]
)

worksheet.format("A1", {
    "textFormat": {
        "bold": True,
        "foregroundColor": {"red": 1, "green": 1, "blue": 1},
        "fontSize": 11
    },
    "backgroundColor": {"red": 0.08, "green": 0.31, "blue": 0.55}
})


# ── Row 2 : Headers ─────────────────────────────────────────────
worksheet.update(
    "A2",
    [header]
)

last_col = chr(64 + len(header))

worksheet.format(f"A2:{last_col}2", {
    "textFormat": {
        "bold": True,
        "foregroundColor": {"red": 1, "green": 1, "blue": 1}
    },
    "backgroundColor": {"red": 0.13, "green": 0.47, "blue": 0.25}
})


# ── Row 3 : Data ────────────────────────────────────────────────
worksheet.update(
    "A3",
    rows,
    value_input_option="USER_ENTERED"
)


# ── Logs ────────────────────────────────────────────────────────
print(f"✅ {len(rows)} rows written successfully!")
print(f"🕒 Last Refreshed: {last_run_eat}")
print(f"📄 Sheet URL: https://docs.google.com/spreadsheets/d/{sheet.id}")

In [ ]:
# ── Cron-style auto refresh every 10 minutes ───────────────────
%pip install apscheduler --quiet

from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from google.oauth2.service_account import Credentials
import gspread

KENYA_TZ = ZoneInfo("Africa/Nairobi")


def write_df_to_sheet(df: pd.DataFrame):
    scope = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive"
    ]

    creds = Credentials.from_service_account_file(
        GOOGLE_CREDENTIALS_PATH,
        scopes=scope
    )
    client = gspread.authorize(creds)

    sheet = client.open(GOOGLE_SHEET_NAME)
    try:
        worksheet = sheet.worksheet(WORKSHEET_TAB)
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sheet.add_worksheet(title=WORKSHEET_TAB, rows=5000, cols=20)

    worksheet.clear()

    header = df.columns.tolist()
    rows = df.fillna("").astype(str).values.tolist()
    last_run_eat = datetime.now(KENYA_TZ).strftime("%Y-%m-%d %H:%M:%S EAT")

    worksheet.update("A1", [[f"Last Refreshed: {last_run_eat} | Total Assets: {len(rows)}"]])
    worksheet.format("A1", {
        "textFormat": {
            "bold": True,
            "foregroundColor": {"red": 1, "green": 1, "blue": 1},
            "fontSize": 11
        },
        "backgroundColor": {"red": 0.08, "green": 0.31, "blue": 0.55}
    })

    worksheet.update("A2", [header])

    last_col = chr(64 + len(header)) if len(header) <= 26 else "Z"
    worksheet.format(f"A2:{last_col}2", {
        "textFormat": {
            "bold": True,
            "foregroundColor": {"red": 1, "green": 1, "blue": 1}
        },
        "backgroundColor": {"red": 0.13, "green": 0.47, "blue": 0.25}
    })

    if rows:
        worksheet.update("A3", rows, value_input_option="USER_ENTERED")

    print(f"✅ {len(rows)} rows written successfully at {last_run_eat}")
    print(f"📄 Sheet URL: https://docs.google.com/spreadsheets/d/{sheet.id}")


def refresh_once():
    print("\n🔄 Running scheduled refresh...")

    mix_creds = get_creds(ACCOUNTS_JSON_PATH, SERVER_KEY)
    api_url = mix_creds["ApiUrl"]
    bearer_token = get_bearer_token(mix_creds)

    raw_positions = get_latest_positions(
        api_url=api_url,
        bearer_token=bearer_token,
        group_ids=GROUP_IDS,
        quantity=QUANTITY,
        cached_since=CACHED_SINCE,
        ensure_reverse_geocoded=ENSURE_REVERSE_GEOCODED,
    )

    asset_lookup = try_asset_endpoints(api_url, bearer_token, GROUP_IDS[0])
    df = parse_positions(raw_positions, asset_lookup)
    write_df_to_sheet(df)


def start_cron_job_every_10_minutes():
    global scheduler

    # Run one immediate refresh now
    refresh_once()

    # Cron expression: */10 * * * *
    scheduler = BackgroundScheduler(timezone=KENYA_TZ)
    scheduler.add_job(
        refresh_once,
        trigger=CronTrigger(minute="*/10"),
        id="positions_to_google_sheet",
        replace_existing=True,
        max_instances=1,
        coalesce=True,
    )
    scheduler.start()

    print("✅ Cron job started: runs every 10 minutes (*/10 * * * *)")
    print("ℹ️ Keep this notebook session running for automatic updates.")


# Start scheduler
start_cron_job_every_10_minutes()